# Streaming

> Streaming helpers for lossless event collation.

In [ ]:
#| default_exp streaming

## Imports

In [ ]:
#| export
import json,copy
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import delegates
from fastspec.errors import *
from fastllm.types import *

In [ ]:
#| hide
import yaml,json
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms
from IPython.core.display import Markdown
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = files('fastllm') / 'specs'
ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.fireworks.ai/inference/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### Delta

Canonical streaming event fragment

In [ ]:
#| export
@dataclass
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    thinking: str = ""
    refusal: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)
    citations: list = field(default_factory=list)
    server_tool_result: dict = None
    finish_reason: str = None
    usage: Usage = None
    raw: dict = field(default_factory=dict)

In [ ]:
#| export
async def norm_and_yield(resp, norm_func):
    async for ev in resp:
        if (d := norm_func(ev)) is not None: yield d

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

`ToolCall` normalizes a tool call with `id`, `name`, and `arguments`, anything extra goes into `extra`

`tool_use` parts are built from tool calls, and store `ToolCall` fields in `data`, which will be converted into correct message type based on provider, and allow users to pass `Msg` directly to the chat history

### PartAccum

Collator for tool calls and other text parts

In [ ]:
#| export
@dataclass
class PartAccum:
    parts: dict[Part|ToolCall] = field(default_factory=dict)
    tool_calls: list[ToolCall] = field(default_factory=list)
    
    def append(self, typ, index, txt='', citations=None, **tc_kwargs):
        'Create and accumulate same type sequential parts'
        if index not in self.parts: 
            if typ==PartType.tool_use: self.parts[index] = ToolCall(**tc_kwargs)
            else:                      self.parts[index] = Part(type=typ, text=txt, data=dict(citations=citations or []))
        else:
            if typ==PartType.tool_use:
                new_args = tc_kwargs.get('arguments', '')
                cur_args = self.parts[index].arguments
                if isinstance(new_args, str) and isinstance(cur_args, str): self.parts[index].arguments += new_args
                elif isinstance(new_args, str) and isinstance(cur_args, dict): self.parts[index].arguments = new_args
                else: self.parts[index].arguments = new_args
            else: 
                self.parts[index].text += txt
                # anthropic citations have matching idx
                self.parts[index].data['citations'].extend(citations or [])
    
    def get_merged(self, with_tools=True):
        tmp_parts = copy.deepcopy(self.parts)
        tool_calls = []
        if with_tools:
            for idx,tc in tmp_parts.items():
                if isinstance(tc, ToolCall):
                    if isinstance(tc.arguments, str): tc.arguments = json.loads(tc.arguments) if tc.arguments else {}
                    tool_calls.append(tc)
                    data = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
                    tmp_parts[idx] = Part(type=PartType.tool_use, data=data)
        
        merged = []
        for p in tmp_parts.values():
            if isinstance(p, ToolCall) and not with_tools: continue
            if merged and merged[-1].type == p.type and p.type in (PartType.text, PartType.thinking): merged[-1].text += p.text
            else: merged.append(p)
        return merged, tool_calls
    
    def finalize(self):
        self.parts, self.tool_calls = self.get_merged()

Incomplete tool calls will cause json error if using `with_tools`

In [ ]:
pa = PartAccum({0: ToolCall(id='toolu_01GF7HEH9s63gdAYL5dbcSj5', name='python', arguments={}, server=False, extra={'caller': {'type': 'direct'}})}, [])
pa.get_merged(False)

([], [])

### mk_acollect_stream

A generic `mk_acollect_stream` function that yields thinking and text (if needed we can yield tool calls to), and collates parts while keeping the order. A custom `index_fn` is used to resolve the index that the current `Delta` event belongs to.

In [ ]:
_fence_back = '`````'
_fence_re = re.compile(f'^{_fence_back}(py|bash)\n(.*?)\n{_fence_back}$', re.DOTALL | re.MULTILINE)

class FenceToolStop:
    def __init__(self, langs): self.langs = langs
    def __call__(self, text):
        "Return trim result if complete fence detected in active lang"
        m = _fence_re.search(text)
        if m and m.group(1) in self.langs: return m.group(0)

In [ ]:
fence_tool = FenceToolStop({'py'})

In [ ]:
sample = 'Here is a Python function that calculates the Fibonacci sequence, along with a simple call example to generate the first 10 numbers:\n\n`````py\ndef fib(n):\n    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    return fib(n-1) + fib(n-2)\n\nfor i in range(10):\n    print(f"fib({i}) = {fib(i)}")\n`````\nThis should be trimmed'

In [ ]:
deltas = [Delta(text=''.join(c)) for c in chunked(sample, 20)]

In [ ]:
list(L(deltas).map(lambda o:nested_idx(o, 'text')))

['Here is a Python fun',
 'ction that calculate',
 's the Fibonacci sequ',
 'ence, along with a s',
 'imple call example t',
 'o generate the first',
 ' 10 numbers:\n\n`````p',
 'y\ndef fib(n):\n    if',
 ' n <= 0:\n        ret',
 'urn 0\n    elif n == ',
 '1:\n        return 1\n',
 '    return fib(n-1) ',
 '+ fib(n-2)\n\nfor i in',
 ' range(10):\n    prin',
 't(f"fib({i}) = {fib(',
 'i)}")\n`````\nThis sho',
 'uld be trimmed']

In [ ]:
part_accum = PartAccum()
for idx, d in enumerate(deltas[:-1]): part_accum.append('text', idx, d.text)

In [ ]:
cur_parts,_ = part_accum.get_merged()

In [ ]:
test_eq(cur_parts[-1].type, PartType.text)
txt = cur_parts[-1].text; txt

'Here is a Python function that calculates the Fibonacci sequence, along with a simple call example to generate the first 10 numbers:\n\n`````py\ndef fib(n):\n    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    return fib(n-1) + fib(n-2)\n\nfor i in range(10):\n    print(f"fib({i}) = {fib(i)}")\n`````\nThis sho'

In [ ]:
fence_tool(txt)

'`````py\ndef fib(n):\n    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    return fib(n-1) + fib(n-2)\n\nfor i in range(10):\n    print(f"fib({i}) = {fib(i)}")\n`````'

In [ ]:
d

Delta(text='i)}")\n`````\nThis sho', thinking='', refusal='', tool_calls=[], citations=[], server_tool_result=None, finish_reason=None, usage=None, raw={})

In [ ]:
#| export
def _trim_delta(d, txt, s):
    "Trim `d.text` so accumulated text in `cur` stops just before stop sequence `s`."
    idx = len(txt) - (txt.find(s) + len(s))
    if idx>0: d.text = d.text[:-idx]

In [ ]:
_trim_delta(d, txt, fence_tool(txt))

In [ ]:
d

Delta(text='i)}")\n`````', thinking='', refusal='', tool_calls=[], citations=[], server_tool_result=None, finish_reason=None, usage=None, raw={})

A `stop_callable` takes the collated text so far, returns a truthy or a full string match which will be used for trimming the current delta:

In [ ]:
#| export
def stop_and_trim(part_accum, d, stop_callables):
    'Stop based on the accumulated text so far, and trim current delta'
    parts,_ = part_accum.get_merged(with_tools=False)
    prev = parts[-1].text if parts and parts[-1].type == PartType.text else ''
    txt = prev + (d.text or '')
    for f in stop_callables:
        if res:=f(txt):
            if isinstance(res, str): _trim_delta(d, txt, res)
            return True
    return False

In [ ]:
deltas = [Delta(text=''.join(c)) for c in chunked(sample, 20)]
part_accum = PartAccum()
for idx, d in enumerate(deltas):
    stop = stop_and_trim(part_accum, d, [fence_tool])
    part_accum.append('text', idx, d.text)
    if stop: break

In [ ]:
test_eq(d.text, 'i)}")\n`````')

In [ ]:
def stop_sequences(seqs):
    "Stop when any sequence appears in the accumulated completion text."
    seqs = L(seqs)
    def _stop(text):
        for s in seqs:
            if s in text: return text[:text.find(s)+len(s)]
    return _stop

In [ ]:
deltas = [Delta(text=''.join(c)) for c in chunked(sample, 20)]
part_accum = PartAccum()
for idx, d in enumerate(deltas):
    stop = stop_and_trim(part_accum, d, [stop_sequences(['Fibonacci sequence'])])
    part_accum.append('text', idx, d.text)
    if stop: break

In [ ]:
test_eq(part_accum.get_merged()[0][0].text, 'Here is a Python function that calculates the Fibonacci sequence')

In [ ]:
#| export
async def mk_acollect_stream(it, index_fn, model=None, api_name=None, vendor_name=None, stop_callables=None):
    "Collect a Delta stream, yielding incremental chunks and a final Completion."
    part_accum,deltas,stop_callables = PartAccum(), [], L(stop_callables)
    fin, usg, typ, last_typ, last_idx = [None]*5
    def _fidx(d, name, pt=None):
        nonlocal typ, last_idx
        typ = getattr(PartType, pt or name)
        idx,last_idx = index_fn(d, typ, last_typ, last_idx)
        return idx
    def _proc(d, name, pt=None, kw='txt', ret=None):
        if not ret and not (val := getattr(d, name)): return
        idx = _fidx(d, name, pt)
        part_accum.append(typ, idx, **(ret or {kw: val}))
        return ret or {name: val}
    def _yield_parts(d):
        for args in [('text',), ('thinking',), ('citations', 'text', 'citations')]:
            if (r := _proc(d, args[0], pt=args[1] if len(args)>1 else None, kw=args[2] if len(args)>2 else 'txt')):
                yield r
    stop, stop_yielded = False, False
    async for d in it:
        # Check stop condition and yield stop delta
        if not stop: stop = stop_and_trim(part_accum, d, stop_callables)
        if stop and not stop_yielded:
            for r in _yield_parts(d): yield r
            stop_yielded = True
        # If stop the remaining deltas are yielded as processing
        if stop: yield {'thinking':'processing'}
        else:
            for r in _yield_parts(d): yield r
        # Rest incl. tools, finish reason, usage is processed independently
        for tc in d.tool_calls:
            args = tc.arguments.get('_delta', tc.arguments)
            _proc(d, 'tool_use', ret=dict(id=tc.id, name=tc.name, arguments=args, server=tc.server, extra=tc.extra))
        if d.server_tool_result:
            idx = _fidx(d, 'server_tool_result')
            part_accum.parts[idx] = Part(type=typ, data=d.server_tool_result)
        if (r:=_proc(d, 'refusal')): yield r
        if d.finish_reason: fin = d.finish_reason
        if d.usage: usg = d.usage
        last_typ = typ
        deltas.append(d)
    part_accum.finalize()
    tcs = part_accum.tool_calls
    if stop: fin = FinishReason.stop
    fin = FinishReason.tool_calls if fin==FinishReason.stop and any(~L(tcs).attrgot('server')) else fin # recheck tool calls post collation
    # tool calls and non-anthropic citations are yielded at the end
    yield Completion(d.raw.get('model', model),
            message=Msg(role="assistant", content=part_accum.parts),
            finish_reason=fin, usage=usg, tool_calls=tcs, api_name=api_name, vendor_name=vendor_name,
            raw={'deltas':deltas})

TODO: yield collated tool calls as ready, e.g. if arguments have ending `}`? To be able to preview tool call status `status_re = re.compile(r'^- ⏳ <code>(.*)</code> ⏳$|^🧠+$', re.MULTILINE)`


Anthropic test:

In [ ]:
deltas = [
    Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text='\n`', thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text='````py\nimport random\n\ndef random_number_generator(start=1, end=100):\n    """Generate a random number between start and end (', thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text='inclusive)."""\n    return random.randint(start, end)\n\nprint(random_number_generator())\n`````\n\nI\'ve written a `random_number_generator` function that', thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text=':\n- Takes two optional parameters: `start` (default: `1`) and `end` (default: `100`)\n- Uses', thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text=" Python's `random.randint()` to return a random integer **between `start` and `end` (inclusive)**\n- Then", thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text=' calls and prints the result\n\nLet me share the number once the execution compl', thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text='etes! 🎲', thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None),
    Delta(text='', thinking='', refusal='', tool_calls=[], citations=[], server_tool_result=None, finish_reason='stop', usage=Usage(prompt_tokens=947, completion_tokens=159, total_tokens=1106, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0))
]

In [ ]:
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    return ifnone(last_idx,0) + 1, ifnone(last_idx,0) + 1

In [ ]:
async def detlas_gen(deltas): 
    for d in deltas: yield d

In [ ]:
async def collect_txt(deltas, **kwargs):
    txt = ''
    async for o in mk_acollect_stream(detlas_gen(deltas), delta_index_fn, **kwargs): 
        if isinstance(o, dict): txt += o.get('text', '')
    return txt

In [ ]:
print(await collect_txt(deltas))


`````py
import random

def random_number_generator(start=1, end=100):
    """Generate a random number between start and end (inclusive)."""
    return random.randint(start, end)

print(random_number_generator())
`````

I've written a `random_number_generator` function that:
- Takes two optional parameters: `start` (default: `1`) and `end` (default: `100`)
- Uses Python's `random.randint()` to return a random integer **between `start` and `end` (inclusive)**
- Then calls and prints the result

Let me share the number once the execution completes! 🎲


In [ ]:
print(await collect_txt(deltas, stop_callables=[fence_tool]))


`````py
import random

def random_number_generator(start=1, end=100):
    """Generate a random number between start and end (inclusive)."""
    return random.randint(start, end)

print(random_number_generator())
`````


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()